This Colab shows some example code how to make use of the
[LiT: Zero-Shot Transfer with Locked-image text Tuning](https://arxiv.org/abs/2111.07991)
models in the `big_vision` codebase.

For more information refer to

https://github.com/google-research/big_vision/blob/main/README.md

https://github.com/google-research/big_vision/blob/main/big_vision/configs/proj/image_text/README.md

### Initialize

In [ ]:
!git clone --branch=main --depth=1 https://github.com/google-research/big_vision
!cd big_vision && git pull

In [ ]:
!pip install -qr big_vision/big_vision/requirements.txt

In [ ]:
import sys
bv_path = './big_vision'
if bv_path not in sys.path:
  sys.path.insert(0, bv_path)

%load_ext autoreload
%autoreload 2

In [ ]:
from absl import flags
from absl import logging
import tensorflow_datasets as tfds
from google.colab import files

logging.set_verbosity(logging.INFO)

def set_max_height(max_height):
  """Limits scrollable area of output cell to `max_height` pixels."""
  import IPython.display
  IPython.display.display(IPython.display.Javascript('''
    google.colab.output.setIframeHeight(0, true, {maxHeight: %d})
  ''' % max_height))

In [ ]:
# Set up Colab  TPUs (if available).
import os
if 'COLAB_TPU_ADDR' in os.environ:
  import jax.tools.colab_tpu
  jax.tools.colab_tpu.setup_tpu()
else:
  !nvidia-smi
import jax
jax.devices()

In [ ]:
# Uncomment this snippet to access a private GCS bucket with prepared
# TFDS datasets.

# import tensorflow_datasets as tfds
# from google.colab import auth
# auth.authenticate_user()  # Required to access access protected GCS buckets.
# import os
# os.environ['TFDS_DATA_DIR'] = 'gs://tensorflow-datasets/datasets'
# builder = tfds.builder('coco_captions')
# b = next(iter(builder.as_dataset('val')))

### Load training run data

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.io import gfile

def plot_metrics(workdir, regexes, cols=4, cmp={}):
  """Plots metrics matching `regexes` from `workdir`."""
  df = pd.DataFrame([json.loads(line) for line in gfile.GFile(f'{workdir}/big_vision_metrics.txt')])
  df = df.set_index('step')
  ms = []
  for regex in regexes:
    for col in df.columns:
      if col not in ms and re.match(regex, col):
        ms.append(col)
  rows = int(np.ceil(len(ms) / cols))
  _, axs = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
  if rows == 1: axs = [axs]
  for i, m in enumerate(ms):
    ax = axs[i // cols][i % cols]
    df[m].dropna().plot(ax=ax)
    if m in cmp: cmp[m].dropna().plot(ax=ax)
    ax.set_title(m)
  plt.tight_layout()
  return df

# Reference run using the tiny 80k "coco-captions" TFDS dataset.
df = plot_metrics('gs://vit_models/lit/big_vision/coco_B16B', [
    'training', 'val/loss', 'img/', '.*net2012',
    '.*cifar100', '.*pet', '.*@1$',
])

### Inference

Using an example from the online Demo

https://google-research.github.io/vision_transformer/lit/

In [ ]:
!test -f apple-ipod.jpg || wget https://cdn.openai.com/multimodal-neurons/assets/apple/apple-ipod.jpg

labels = [
    'an apple',
    'an ipod',
    'granny smith',
    'an apple with a note saying "ipod"',
    'an adversarial attack',
]

import PIL
import numpy as np
img = np.array(PIL.Image.open('apple-ipod.jpg'))
import matplotlib.pyplot as plt
plt.imshow(img)
img.shape, img.dtype

In [ ]:
!test -f LiT-B16B.npz || gsutil cp gs://vit_models/lit/LiT-B16B.* .

In [ ]:
files.view('big_vision/big_vision/configs/proj/image_text/siglip_lit_coco.py')
from big_vision.configs.proj.image_text import siglip_lit_coco as lit_coco
arg = 'txt=bert_base,img=B/16,img_head,init=LiT-B16B.npz'
config = lit_coco.get_config(arg)

In [ ]:
# Initialize template params...
import importlib
import jax.numpy as jnp

model_mod = importlib.import_module(f'big_vision.models.{config.model_name}')

model = model_mod.Model(**config.model)

init_params = [
    jnp.zeros(shape, dtype)
    for shape, dtype in zip(config.init_shapes, config.init_types)
]

params0 = model.init(jax.random.PRNGKey(42), *init_params)['params'].unfreeze()

In [ ]:
# ... and load/modify pre-trained params.
from big_vision import utils
# Note that `.load()` is responsible for parameter tree surgery to adapt old
# checkpoints to most recent source code.
params = model_mod.load(params0, 'LiT-B16B.npz', config.model)

In [ ]:
# The preprocessing is optimized for efficiently streaming through TFDS
# datasets - below code runs it separately on the image and every text.
set_max_height(222)

from big_vision.pp import builder as pp_builder
for pp_mod in config.pp_modules:
  importlib.import_module(f'big_vision.pp.{pp_mod}')

pp_str = config.evals.val.pp_fn.replace('decode|', '')
imgs = np.array(pp_builder.get_preprocess_fn(pp_str)({
    'image': img[None],
    'captions/text': np.array(['']),
})['image'])
txts = np.stack([
    pp_builder.get_preprocess_fn(pp_str, log_data=False)({
        'image': img[None],
        'captions/text': np.array([label]),
    })['labels']
    for label in labels
])
imgs.shape, txts.shape

In [ ]:
%debug

In [ ]:
zimg, _, _ = model.apply({'params': params}, imgs, None)
_, ztxt, _ = model.apply({'params': params}, None, txts)

In [ ]:
probs = jax.nn.softmax((zimg[0] @ ztxt.T * np.exp(params['t'])))
list(zip(labels, probs.tolist()))

### Run evaluation

Below code runs a minimal version of the `big_vision.tols.eval_only` script.

In [ ]:
files.view('big_vision/big_vision/tools/eval_only.py')
from big_vision.tools import eval_only

In [ ]:
!test -f LiT-B16B.npz || gsutil cp gs://vit_models/lit/LiT-B16B.* .

In [ ]:
from big_vision.configs.proj.image_text import lit_coco
arg = 'txt=bert_base,img=B/16,img_head,init=LiT-B16B.npz'
config = lit_coco.get_config(arg)

In [ ]:
# From all the pre-defined evaluators...
set_max_height(222)
config.evals

In [ ]:
# ... run a single zeroshot discriminative classifier on pets.
config.evals = {
    'disclf': {
        **config.evals.disclf,
        'dataset_names': ['oxford_iiit_pet'],
        'dataset_overrides': (),
    },
}
config.evals

In [ ]:
# Prepare pets dataset.
tfds.builder('oxford_iiit_pet').download_and_prepare()

In [ ]:
workdir = 'lit_coco_B16B_eval'
!mkdir $workdir
flags.FLAGS.workdir = workdir
config.input.batch_size = 512
flags.FLAGS.config = config

In [ ]:
# Should run in ~5 minutes on a T4 GPU...
set_max_height(444)
eval_only.main([])

In [ ]:
# ... and yield a final 81% accuracy.
import json
json.loads(open(flags.FLAGS.workdir + '/big_vision_metrics.txt').readline())